# D092 — Operator Overloading

QuickCart performs operations on monetary values: add delivery charges, subtract discounts, reverse a transaction, and compare prices. Operator overloading lets these domain objects use familiar Python operators while keeping their business rules inside the class.

## What is operator overloading?

The same operator already behaves differently for different built-in types:

```python
10 + 20                 # numeric addition
"Quick" + "Cart"       # string concatenation
[1, 2] + [3, 4]         # list concatenation
```

A user-defined class can define the meaning of an operator through Python's **special methods**, also called **dunder methods** because their names begin and end with double underscores.

Python translates an operator expression into a method call:

| Expression | Special-method call |
|---|---|
| `+value` | `value.__pos__()` |
| `-value` | `value.__neg__()` |
| `left + right` | `left.__add__(right)` |
| `left - right` | `left.__sub__(right)` |
| `left == right` | `left.__eq__(right)` |
| `left < right` | `left.__lt__(right)` |

Code normally uses the operator, not a direct dunder-method call.

## A monetary value object

Using plain numbers for e-commerce amounts can accidentally combine different currencies. A `Money` object keeps the amount and currency together.

`Decimal` is used instead of `float` because decimal financial values must not acquire binary floating-point rounding artifacts.

In [ ]:
from decimal import Decimal


class Money:
    def __init__(self, amount, currency="INR"):
        self.amount = Decimal(str(amount))
        self.currency = currency.upper()

    def __str__(self):
        return f"{self.currency} {self.amount:,.2f}"

    def __repr__(self):
        return f"Money(amount={str(self.amount)!r}, currency={self.currency!r})"


product_price = Money("3499.00")
print(product_price)
print(repr(product_price))

# 1. Unary operators

A unary operator works on one object.

- Unary `+` calls `__pos__` and returns the value with its current sign.
- Unary `-` calls `__neg__` and reverses the sign.

Negating money is useful when representing a refund, reversal, or accounting adjustment. These methods return new objects; they do not modify the original amount.

In [ ]:
class Money:
    def __init__(self, amount, currency="INR"):
        self.amount = Decimal(str(amount))
        self.currency = currency.upper()

    def __str__(self):
        return f"{self.currency} {self.amount:,.2f}"

    def __repr__(self):
        return f"Money(amount={str(self.amount)!r}, currency={self.currency!r})"

    def __pos__(self):
        return Money(self.amount, self.currency)

    def __neg__(self):
        return Money(-self.amount, self.currency)


payment = Money("2499.00")
refund = -payment

print(+payment)
print(refund)
print(payment)  # Original object is unchanged

# 2. Binary `+` and `-`

A binary operator works with a left operand and a right operand.

```python
item_price + delivery_charge
order_total - discount
```

`__add__(self, other)` implements `self + other`; `__sub__(self, other)` implements `self - other`.

Only amounts in the same currency can be combined. If the currencies differ, the operation raises a clear business error. If `other` is not a `Money` object, the method returns `NotImplemented`, allowing Python to try the other operand's reflected operation or raise an appropriate `TypeError`.

In [ ]:
class Money:
    def __init__(self, amount, currency="INR"):
        self.amount = Decimal(str(amount))
        self.currency = currency.upper()

    def __str__(self):
        return f"{self.currency} {self.amount:,.2f}"

    def __repr__(self):
        return f"Money(amount={str(self.amount)!r}, currency={self.currency!r})"

    def _require_same_currency(self, other):
        if self.currency != other.currency:
            raise ValueError("money values must use the same currency")

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        self._require_same_currency(other)
        return Money(self.amount + other.amount, self.currency)

    def __sub__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        self._require_same_currency(other)
        return Money(self.amount - other.amount, self.currency)


item_price = Money("3499.00")
delivery_charge = Money("100.00")
discount = Money("250.00")

gross_total = item_price + delivery_charge
net_total = gross_total - discount

print(gross_total)
print(net_total)
print(item_price)  # Binary operations also return new objects

## Invalid binary operations

Operator overloading should reject operations that have no meaningful interpretation. Adding INR directly to USD without an exchange rate is invalid, and adding a `Money` object directly to a string is unsupported.

In [ ]:
try:
    print(Money("100", "INR") + Money("10", "USD"))
except ValueError as error:
    print(type(error).__name__, "-", error)

try:
    print(Money("100") + "delivery")
except TypeError as error:
    print(type(error).__name__, "-", error)

# 3. Object equality

Equality and identity answer different questions:

- `left is right` asks whether both variables refer to the exact same object.
- `left == right` asks whether both objects should be considered equal in value.

Without a custom `__eq__`, user-defined objects normally compare by identity. Two separate `Money("499")` objects contain equivalent data but are not the same object.

In [ ]:
class BasicMoney:
    def __init__(self, amount, currency="INR"):
        self.amount = Decimal(str(amount))
        self.currency = currency


first = BasicMoney("499")
second = BasicMoney("499")
same_reference = first

print(first == second)       # No value-based __eq__ yet
print(first is second)
print(first is same_reference)

## `__eq__`: Python's equality method

Python does not use a standard Java-style `equals()` method. The `==` operator calls `__eq__`:

```python
left == right
# conceptually becomes
left.__eq__(right)
```

For money, equality requires both the amount and currency to match. `__eq__` returns `NotImplemented` for an unrelated type so Python can handle the comparison correctly. In modern Python, `!=` uses the logical opposite of `__eq__` unless `__ne__` supplies different behaviour.

In [ ]:
class Money:
    def __init__(self, amount, currency="INR"):
        self.amount = Decimal(str(amount))
        self.currency = currency.upper()

    def __repr__(self):
        return f"Money(amount={str(self.amount)!r}, currency={self.currency!r})"

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return self.amount == other.amount and self.currency == other.currency


listed_price = Money("499.00", "INR")
checkout_price = Money("499", "inr")
usd_price = Money("499", "USD")

print(listed_price == checkout_price)
print(listed_price is checkout_price)
print(listed_price == usd_price)
print(listed_price != usd_price)
print(listed_price == "INR 499")

## Calling equality as a function

The `operator` module provides function forms of Python operators. `operator.eq(left, right)` performs the same equality operation as `left == right` and therefore invokes the overloaded equality behaviour.

A direct call to `left.__eq__(right)` is useful for learning or debugging, but application code should normally use `==` or `operator.eq`.

In [ ]:
import operator


first_price = Money("1299", "INR")
second_price = Money("1299.00", "INR")

print(first_price == second_price)
print(operator.eq(first_price, second_price))
print(first_price.__eq__(second_price))  # Educational direct call

# 4. Relational operators

Relational operators compare ordering:

| Operator | Method |
|---|---|
| `<` | `__lt__` |
| `<=` | `__le__` |
| `>` | `__gt__` |
| `>=` | `__ge__` |

Money values can be ordered only when their currencies match. The implementation below validates the other operand and currency in one helper method, then each comparison remains small and clear.

In [ ]:
class Money:
    def __init__(self, amount, currency="INR"):
        self.amount = Decimal(str(amount))
        self.currency = currency.upper()

    def __str__(self):
        return f"{self.currency} {self.amount:,.2f}"

    def _comparable_amount(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        if self.currency != other.currency:
            raise ValueError("money values must use the same currency")
        return other.amount

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return self.amount == other.amount and self.currency == other.currency

    def __lt__(self, other):
        other_amount = self._comparable_amount(other)
        if other_amount is NotImplemented:
            return NotImplemented
        return self.amount < other_amount

    def __le__(self, other):
        other_amount = self._comparable_amount(other)
        if other_amount is NotImplemented:
            return NotImplemented
        return self.amount <= other_amount

    def __gt__(self, other):
        other_amount = self._comparable_amount(other)
        if other_amount is NotImplemented:
            return NotImplemented
        return self.amount > other_amount

    def __ge__(self, other):
        other_amount = self._comparable_amount(other)
        if other_amount is NotImplemented:
            return NotImplemented
        return self.amount >= other_amount


sale_price = Money("2799")
regular_price = Money("3499")

print(sale_price < regular_price)
print(sale_price <= regular_price)
print(regular_price > sale_price)
print(regular_price >= Money("3499"))

## Relational operators enable sorting

`sorted()` uses the objects' ordering behaviour. Once `Money.__lt__` is available, a collection of same-currency prices can be sorted naturally.

In [ ]:
prices = [Money("3499"), Money("799"), Money("18999"), Money("1499")]

for price in sorted(prices):
    print(price)

# 5. Complete example

The final class combines unary, arithmetic, equality, and relational operators. Each operator returns a new `Money` object or a Boolean and preserves the same-currency business rule.

In [ ]:
class Money:
    def __init__(self, amount, currency="INR"):
        self.amount = Decimal(str(amount))
        self.currency = currency.upper()

    def __str__(self):
        return f"{self.currency} {self.amount:,.2f}"

    def __repr__(self):
        return f"Money(amount={str(self.amount)!r}, currency={self.currency!r})"

    def _comparable_amount(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        if self.currency != other.currency:
            raise ValueError("money values must use the same currency")
        return other.amount

    def __pos__(self):
        return Money(self.amount, self.currency)

    def __neg__(self):
        return Money(-self.amount, self.currency)

    def __add__(self, other):
        other_amount = self._comparable_amount(other)
        if other_amount is NotImplemented:
            return NotImplemented
        return Money(self.amount + other_amount, self.currency)

    def __sub__(self, other):
        other_amount = self._comparable_amount(other)
        if other_amount is NotImplemented:
            return NotImplemented
        return Money(self.amount - other_amount, self.currency)

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return self.amount == other.amount and self.currency == other.currency

    def __lt__(self, other):
        other_amount = self._comparable_amount(other)
        if other_amount is NotImplemented:
            return NotImplemented
        return self.amount < other_amount

    def __le__(self, other):
        other_amount = self._comparable_amount(other)
        if other_amount is NotImplemented:
            return NotImplemented
        return self.amount <= other_amount

    def __gt__(self, other):
        other_amount = self._comparable_amount(other)
        if other_amount is NotImplemented:
            return NotImplemented
        return self.amount > other_amount

    def __ge__(self, other):
        other_amount = self._comparable_amount(other)
        if other_amount is NotImplemented:
            return NotImplemented
        return self.amount >= other_amount


subtotal = Money("5000")
delivery = Money("100")
coupon = Money("500")
amount_paid = Money("4600.00")

order_total = subtotal + delivery - coupon

print("Order total:", order_total)
print("Payment complete:", amount_paid == order_total)
print("Coupon is smaller than subtotal:", coupon < subtotal)
print("Refund entry:", -order_total)

## Final recap

- Operators call special methods such as `__add__`, `__neg__`, and `__eq__`.
- Unary operators work on one object; binary operators combine two operands.
- `==` checks value equality through `__eq__`; `is` checks object identity.
- Python's equality method is `__eq__`, not a standard `equals()` method.
- Relational methods make domain objects comparable and sortable.
- Return `NotImplemented` for unsupported operand types.
- Raise a clear exception when operand types are supported but the business operation is invalid.
- Overload an operator only when its meaning is natural and unsurprising for the class.